# Nightjar Music Generation - based on RAVE

In this Notebook, we will generate music based on the song "Nightjar" by Cosmo Sheldrake. 

For this, we will apply a previously trained RAVE-model (see ``rave/``).


Currently, we have the models ``epoch_2000000.ckpt.ts``, ``best.ckpt.ts`` available.

- The name is given by the fact that RAVE creates checkpoints during training after a certain number of steps/epochs. The model ``epoch_2000000.ckpt.ts`` was trained with 2,000,000 RAVE steps (approx 13k epochs). ``best.ckpt.ts`` is continuously updated and corresponds to the model with smallest validation loss.
- The model was trained on the training data provided in `rave/data/training_data`. It includes bird/nature sounds, the Nightjar song itself, as well as more songs by Cosmo Sheldrake for more musical inspiration. Data augmentation was applied: The model was trained using 20 seconds audio files; mono; 441000Htz. To increase the number of song snippets for training, offset of 0/5/10 seconds was used.


## How is the music generated?

The basic idea is to interpolate between points of existing Nightjar snippets in the latent space. Along the line between those two points, we take multiple points. By decoding them from latent space back to real audio, we will generate 'new' music.

The goal is to make a transition from the original Nightjar song to the generated song. Our generated song will be around 3 minutes long. We decided that the first part will be bird/nature sounds, since this is how the original Nightjar ends (its last ~17seconds are only few bird calls). We will also end the song with bird sounds, to generate a loop for the Nightjar project. 

Our song will therefore look as follows:


```text
Part 1  nature/bird sounds intro
Part 2  nightjar-based music; wandering around latent space, using nightjar (and similar song) points
Part 4  more experimental part; we move further away from latent space
Part 3  back to nature/birds; outro
```

## 0. Installation


In [ ]:
# pip install -r requirements.txt

## 1. Imports and configuration


In [ ]:
from pathlib import Path
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import librosa
import soundfile as sf

from tqdm.auto import tqdm
from IPython.display import Audio, display, Markdown

from sklearn.decomposition import PCA

import torch
from torchinfo import summary


import plotly.express as px

import json
from matplotlib.ticker import FuncFormatter

import plotly.express as px
import plotly.graph_objects as go

warnings.filterwarnings("ignore")


In [ ]:
# Paths to data and model
RAVE_MODEL_PATH = Path("models/epoch_2000000.ckpt.ts") # Path("best.ckpt.ts"), Path("epoch_2000000.ckpt.ts")

TRAINING_DATA_DIR = Path("data/training_data")
TRAINING_DATA_BIRDS_DIR = TRAINING_DATA_DIR / "birds"
TRAINING_DATA_DIR_SONGS = TRAINING_DATA_DIR / "songs"

OUTPUT_DIR = Path("data/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Audio and RAVE settings 
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SR = 44100 # sample rate (used in training data)

# Generation settings
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

## 2. Load the RAVE model


In [ ]:
if not RAVE_MODEL_PATH.exists():
    raise FileNotFoundError(f"RAVE model not found: {RAVE_MODEL_PATH}")

rave = torch.jit.load(str(RAVE_MODEL_PATH), map_location=DEVICE)
rave.eval()

summary(rave)

## Modeling Metrics

The RAVE model was trained on the OTH KI-GPU Server for faster and longer training compared to local training.

The model metrics can be analyzed and downloaded in TensorBoard.


### RAVE Training
RAVE's training process has two distinct phases :
- **Phase 1 : auto-encoding phase** - The encoder and the decoder of the model are trained jointly on a spectral loss
- **Phase 2 : adversarial phase** - The encoder is frozen and a discriminator is introduced ; the decoder is fine-tuned with the discriminator using an adversarial training setup

### Monitoring phase 1 
The encoder and the decoder are trained using three losses : 
- the fullband_spectral_distance
- multiband_spectral_distance
- regularization

Typically, the two first losses should be regularaly decreasing during this first phase. Losses are plotted by batch, so the curves may be be a little noisy.

Default regularization was applied. This regularization term is the difference between the latent space and an isotropic gaussian. A zero regularization typically means that your latent space is random noise, a degenerated behaviour. Reversely, a very high regularization term means that your latent space is very "rough", or bursted around undesirable values.

### Monitoring phase 2
Phase 2 usually starts after 1Mio RAVE-steps. The training drastically changes: adversarial training comes into play! Therefore, the adversarial loss can be viewed from this phase.

Source: https://forum.ircam.fr/article/detail/training-rave-models-on-custom-data/

In [ ]:
window_size = 20
phase_boundary = 1_000_000

plots = [
    ("metrics_json/validation.json", "validation", (4.8,7)),
    ("metrics_json/fullband_spectral_distance.json", "fullband_spectral_distance", (3.5, 9)),
    ("metrics_json/multiband_spectral_distance.json", "multiband_spectral_distance", (3.5, 9)),
    ("metrics_json/regularization.json", "regularization", (0.5, 1.5)),
    ("metrics_json/adversarial.json", "adversarial", (1,5)),
]

fig, axes = plt.subplots(5, 1, figsize=(14, 12), sharex=True, constrained_layout=True)

for ax, (file_path, name, ylim) in zip(axes, plots):
    with open(file_path, "r") as f:
        df = pd.DataFrame(json.load(f), columns=["timestamp", "step", "value"])

    df["smooth"] = df["value"].rolling(window_size, center=True, min_periods=1).mean()

    ax.axvspan(df["step"].min(), phase_boundary, color="yellow", alpha=0.25, label="Phase 1")
    ax.axvspan(phase_boundary, df["step"].max(), color="orange", alpha=0.25, label="Phase 2")
    ax.plot(df["step"], df["value"], linewidth=1, alpha=0.3, label=name)
    ax.plot(df["step"], df["smooth"], linewidth=2.5, color="black", label=f"Smoothed {name}")
    ax.axvline(phase_boundary, color="black", linestyle="--", linewidth=2, label="1 mio steps")

    ax.text(500_000, 0.88, "Phase 1", transform=ax.get_xaxis_transform(), ha="center", va="top", fontsize=16)
    ax.text(1_500_000, 0.88, "Phase 2", transform=ax.get_xaxis_transform(), ha="center", va="top", fontsize=16)

    ax.set(title=name, ylabel=name, ylim=ylim)
    ax.grid(True)
    ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1))

axes[-1].set_xlabel("Steps (mio)")
axes[-1].xaxis.set_major_formatter(FuncFormatter(lambda x, pos: f"{x / 1_000_000:.1f}"))

plt.show()

The first plot shows that the validation loss is lower during the first training phase than during the second phase. This is also reflected in the checkpoint selection: RAVE saves a `best` model based on the lowest validation loss. In this run, the best checkpoint was saved at step 965,888 (corresponding to epoch 6271), which matches the minimum region in the plot.

After 1 million steps, RAVE enters the **adversarial training phase**. At this point, the spectral losses increase, which is expected because the training objective changes: a discriminator is introduced and the decoder is fine-tuned adversarially. The adversarial loss also increases and fluctuates strongly. A higher adversarial loss can indicate that the decoder has more difficulty fooling the discriminator, but it is not a direct measure of perceived audio quality. The model was afterwards trained again on additional 2Mio steps. The adversarial loss did not decrease. This indicates that the model either needs a lot more time for training, or, what is more likely, that our training data wasnt enough, with 144mins of audio footage.

Based on the validation loss alone, the `best` checkpoint would be the stronger choice, since it has the lowest reconstruction-based validation score. However, the 2M-step model is still relevant because it has gone through the adversarial phase. Training to 2 million steps took about 2.5 days, and longer training might change the adversarial behavior, although this is not guaranteed.

In practice, the generated audio from the `best` model and the 2M-step model sounded almost identical for the generated audio. Therefore, while the `best` checkpoint is preferable according to the validation metric, using the 2M-step model is also reasonable, especially if the goal is to discuss or demonstrate the adversarially fine-tuned version of RAVE.


## 3. Load audio files

We are loading the training data and give each file a label:

- `bird_training`: prepared bird/nature training snippets (20 seconds max)
- `song_training_nightjar`: prepared Nightjar training snippets (20 seconds max)
- `song_training_similar`: prepared similar-song training snippets (20 seconds max)

In [ ]:
# label training data
audio_files, file_labels = [], []

for p in sorted(TRAINING_DATA_BIRDS_DIR.rglob("*")):
    audio_files.append(p)
    file_labels.append("bird_training")

for p in sorted(TRAINING_DATA_DIR_SONGS.rglob("*")):
    audio_files.append(p)
    if "nightjar" in p.stem.lower():
        file_labels.append("song_training_nightjar")
    else:
        file_labels.append("song_training_similar")

print("Files found:", len(audio_files))
print(pd.Series(file_labels).value_counts())


In [ ]:
# Load audio files
audio_cache = {}

for path in tqdm(audio_files, desc="Loading audio"):
    y, sr = librosa.load(path, sr=SR, mono=True)
    audio_cache[str(path)] = y

print("Loaded audio files:", len(audio_cache))

## 4. Helper functions


In [ ]:
# Encode audio with RAVE
@torch.no_grad()
def rave_encode(y):
    dtype = next(rave.parameters()).dtype
    x = torch.tensor(y, device=DEVICE, dtype=dtype).view(1, 1, -1)
    return rave.encode(x).squeeze(0).cpu().numpy()

# Decode latent representation with RAVE
@torch.no_grad()
def rave_decode(z):
    dtype = next(rave.parameters()).dtype
    z = torch.tensor(z, device=DEVICE, dtype=dtype).view(1, 8, -1)
    return rave.decode(z).squeeze().cpu().numpy()

# Display audio
def show_audio(title, y):
    y = np.asarray(y, dtype=np.float32)

    display(Markdown(f"### {title}"))
    display(Audio(y, rate=SR))

# Summarize a latent sequence into one vector
# Input shape: (8 latent dimensions, time_steps)
# Output: 8 dimensions × 4 statistics: mean, std, p10, p90
def latent_summary(z):
    z = np.asarray(z)

    return np.concatenate([
        z.mean(axis=1),
        z.std(axis=1),
        np.percentile(z, 10, axis=1),
        np.percentile(z, 90, axis=1),
    ])

In [ ]:
test_file = audio_files[0]

file_audio, file_sr = sf.read(str(test_file), dtype="float32")
cached_audio = audio_cache[str(test_file)]

if file_audio.ndim > 1:
    file_audio = file_audio[:, 0]

print("file samples:", len(file_audio))
print("file duration:", len(file_audio) / file_sr)

print("cached samples:", len(cached_audio))
print("cached duration:", len(cached_audio) / SR)

## 5. Quick RAVE reconstruction test

- Check if audio is loaded correctly
- Encode and then decode files --> how does the reconstruction sound?

In [ ]:
for label in ["bird_training", "song_training_nightjar", "song_training_similar"]:
    candidate_indices = [
        i for i, lab in enumerate(file_labels)
        if lab == label
    ]

    print()
    print(f"=== Example for label: {label} ===")

    if not candidate_indices:
        print("No files found for this label.")
        continue

    test_i = random.choice(candidate_indices)
    test_file = audio_files[test_i]
    test_audio = audio_cache[str(test_file)]

    z_test = rave_encode(test_audio)
    recon_test = rave_decode(z_test)

    print("Selected file:", test_file)
    print("z_test shape:", z_test.shape)

    show_audio("Original test audio", test_audio)
    show_audio("RAVE reconstruction", recon_test)

The reconstruction is a bit noisy but overall it works well!

## 6. Encode all audio into RAVE latent space

We encode our training data to see how the latent space looks like.

In [ ]:
z_list = []
summary_list = []
rows = []

for path, label in tqdm(list(zip(audio_files, file_labels)), desc="Encoding audio with RAVE"):
    path_str = str(path)

    # Load the audio file
    y = audio_cache[path_str]

    # Encode audio with RAVE
    z = rave_encode(y)

    # Store latent data
    z_list.append(z)
    summary_list.append(latent_summary(z))

    # Store metadata
    rows.append({
        "file": path_str,
        "source_type": label,
        "z_shape": str(tuple(z.shape)),
        "audio_seconds": len(y) / SR,
    })

metadata = pd.DataFrame(rows)

Z_summary = np.stack(summary_list)

print("Encoded files:", len(metadata))
print(metadata["source_type"].value_counts())
print("Z_summary shape:", Z_summary.shape)

metadata.to_csv(OUTPUT_DIR / "rave_latent_metadata.csv", index=False)

In [ ]:
# Check shapes of all latent sequences
shapes = [z.shape for z in z_list]

unique_shapes = sorted(set(shapes))

print("Number of latent sequences:", len(z_list))
print("Number of unique shapes:", len(unique_shapes))
print("Unique shapes:")

for shape in unique_shapes:
    count = shapes.count(shape)
    print(f"  {shape}: {count} files")

Each latent sequence has a shape like `(8, time_steps)`. For example, `(8, 431)`. 
This means that the model represents the audio with **8 latent dimensions** over **431 time steps**.

The latent representation is therefore not a single fixed vector. Instead, it is a time-varying representation: for every time step, RAVE stores 8 latent values that describe the audio at that point in time.

## 7. Visualizing the latent space with PCA

Next, we visualize the latent space learned by the RAVE model. Since the input data is audio, each file is not encoded as a single point, but as a sequence over time. For one 20-second file, the latent representation has shape `(8, 431)`: 8 latent dimensions across 431 time steps.

Because of this structure, we use two PCA views:

- First, we apply PCA directly to the raw latent time steps. 

    - Each encoded audio file has a latent representation with shape `(8, time_steps)`. The first number, `8`, is the number of latent dimensions learned by the RAVE model. The second number, `time_steps`, is the number of latent time steps for one (20-second) training audio file.

    - For direct PCA, each time step is treated as one point in the original 8-dimensional latent space. This means that one audio file contributes 431 points to the PCA plot, and each point contains 8 values. These 8 values describe the state of the encoded audio at one short moment in time.

- Second, we create a summarized PCA.
    - Here, each point represents one complete 20-second audio file.
    - For comparing complete sounds, each latent point `(8, time_steps)` is summarized into one fixed-length feature vector per file.
    - For each of the 8 latent dimensions, four statistics are calculated: the mean, standard deviation, 10th percentile, and 90th percentile. This results in `8 × 4 = 32` summary features per file.
    - PCA is then applied to these 32-dimensional summary vectors. Unlike the direct latent PCA, where each point represents one short moment in time, this plot shows one point per full audio sample. This makes it easier to compare bird recordings, similar music samples, and nightjar-related music samples at the level of complete sounds.

### a) PCA of original latent space --> one point = one time step of one audio sample

In [ ]:
# Each encoded audio file has a latent representation z with shape:
#     (latent_dimensions, time_steps)
#
# In this notebook, this is for example:
#     (8, 431)
#
# For PCA, each row should be one observation.
# Therefore, we transpose each latent array from:
#     (8, 431)
# to:
#     (431, 8)
#
# After this transformation:
#     one row = one latent time step
#     one column = one RAVE latent dimension

all_latent_steps = []
latent_step_info = []

for z, row in zip(z_list, metadata.to_dict("records")):
    file_path = row["file"]
    source_type = row["source_type"]

    # transpose latent point
    latent_steps = z.T

    # Add all latent time steps from this file to the PCA input
    all_latent_steps.append(latent_steps)

    # Store metadata for each latent time step
    for time_step in range(latent_steps.shape[0]):
        latent_step_info.append({
            "file": file_path,
            "file_label": Path(file_path).name,
            "source_type": source_type,
            "latent_time_step": time_step,
        })

# Combine all files into one matrix for PCA
Z_latent = np.vstack(all_latent_steps)
latent_metadata = pd.DataFrame(latent_step_info)

print("Direct latent PCA input shape:", Z_latent.shape)
print("Rows:", Z_latent.shape[0], "latent time steps")
print("Columns:", Z_latent.shape[1], "RAVE latent dimensions")

In [ ]:
print("Latent Dataset:", Z_latent)

In [ ]:
for i in range(5):
    print("Dimension of snippet", i , ":", all_latent_steps[i].shape)

print("Latent Data for snippet 0:")
print(all_latent_steps[0])

In [ ]:
print("The first snippet has 431 time steps.")
latent_metadata[:431]

In [ ]:
# Apply PCA directly to the RAVE latent values

pca_latent = PCA(n_components=2, random_state=RANDOM_SEED)
Z_latent_2d = pca_latent.fit_transform(Z_latent)

latent_metadata["latent_pca_1"] = Z_latent_2d[:, 0]
latent_metadata["latent_pca_2"] = Z_latent_2d[:, 1]

print("Direct latent PCA explained variance:")
print(
    f"PC1: {pca_latent.explained_variance_ratio_[0]:.3f} "
    f"({pca_latent.explained_variance_ratio_[0] * 100:.1f}%)"
)
print(
    f"PC2: {pca_latent.explained_variance_ratio_[1]:.3f} "
    f"({pca_latent.explained_variance_ratio_[1] * 100:.1f}%)"
)

In [ ]:
# Plot the PCA result

plot_latent_df = latent_metadata.copy()

fig_latent = px.scatter(
    plot_latent_df,
    x="latent_pca_1",
    y="latent_pca_2",
    color="source_type",
    hover_data={
        "source_type": True,
        "latent_pca_1": ":.3f",
        "latent_pca_2": ":.3f",
    },
    title="Direct PCA on RAVE latent time steps",
    width=950,
    height=700,
    opacity=0.45,
    render_mode="webgl",
)

for trace in fig_latent.data:
    trace.update(
        marker=dict(
            size=3,
            opacity=0.45,
        )
    )

fig_latent.show()


# Plot the zoomed direct PCA result
# Same data, just zoomed.

plot_latent_df = latent_metadata.copy()

fig_latent = px.scatter(
    plot_latent_df,
    x="latent_pca_1",
    y="latent_pca_2",
    color="source_type",
    hover_data={
        "source_type": True,
        "latent_pca_1": ":.3f",
        "latent_pca_2": ":.3f",
    },
    title="Direct PCA on RAVE latent time steps (zoomed)",
    width=950,
    height=700,
    opacity=0.45,
    render_mode="webgl",
)

for trace in fig_latent.data:
    trace.update(
        marker=dict(
            size=2.5,
            opacity=0.35,
        )
    )

fig_latent.update_xaxes(range=[-4, 4])
fig_latent.update_yaxes(range=[-4, 4])

fig_latent.show()

There are a few outliers, but overall the distributions is centered at (0,0). There is no strong clustering between birds/songs, rather ... TODO

### b) PCA of summarized latent space --> one point = one summary for all time steps of one audio sample

For each of the 8 latent dimensions, we compute:
-  mean              -> average position of the latent dimension
- standard deviation -> how much the latent dimension moves over time
- 10th percentile    -> low-end value, ignoring extreme minimum outliers
- 90th percentile    -> high-end value, ignoring extreme maximum outliers


In [ ]:
# Each encoded audio file has a latent representation z with shape:
#     (latent_dimensions, time_steps)
#
# In this notebook, this is for example:
#     (8, 431)
#
# For the summarized PCA, we want one point per complete audio file.
# Therefore, we summarize the time dimension of each latent sequence.
#
# For each of the 8 latent dimensions, we compute:
#     mean              -> average position of the latent dimension
#     standard deviation -> how much the latent dimension moves over time
#     10th percentile    -> low-end value, ignoring extreme minimum outliers
#     90th percentile    -> high-end value, ignoring extreme maximum outliers
#
# This converts each file from:
#     (8, 431)
# to:
#     (32,)
#
# because:
#     8 latent dimensions × 4 statistics = 32 summary features
#
# After this transformation:
#     one row = one complete audio file
#     one column = one summary feature

summary_rows = []

for z in z_list:
    z = np.asarray(z)

    z_summary = np.concatenate([
        z.mean(axis=1),
        z.std(axis=1),
        np.percentile(z, 10, axis=1),
        np.percentile(z, 90, axis=1),
    ])

    summary_rows.append(z_summary)

# Combine all file summaries into one matrix for PCA
Z_summary = np.vstack(summary_rows)

print("Summarized PCA input shape:", Z_summary.shape)
print("Rows:", Z_summary.shape[0], "audio files")
print("Columns:", Z_summary.shape[1], "latent summary features")

In [ ]:
# Apply PCA to the summarized latent features
#
# No scaling is applied here, so PCA uses the original numeric scale
# of the summary features.

pca_summary = PCA(n_components=2, random_state=RANDOM_SEED)
Z_summary_2d = pca_summary.fit_transform(Z_summary)

metadata["summary_pca_1"] = Z_summary_2d[:, 0]
metadata["summary_pca_2"] = Z_summary_2d[:, 1]

print("Summarized latent PCA explained variance without scaling:")
print(
    f"PC1: {pca_summary.explained_variance_ratio_[0]:.3f} "
    f"({pca_summary.explained_variance_ratio_[0] * 100:.1f}%)"
)
print(
    f"PC2: {pca_summary.explained_variance_ratio_[1]:.3f} "
    f"({pca_summary.explained_variance_ratio_[1] * 100:.1f}%)"
)

In [ ]:
# Plot the summarized PCA result
#
# In this plot, each point represents one complete 20-second audio file.

plot_summary_df = metadata.copy()

if "file_label" not in plot_summary_df.columns:
    plot_summary_df["file_label"] = plot_summary_df["file"].astype(str).apply(lambda x: Path(x).name)

fig_summary = px.scatter(
    plot_summary_df,
    x="summary_pca_1",
    y="summary_pca_2",
    color="source_type",
    hover_name="file_label",
    hover_data={
        "file": True,
        "source_type": True,
        "audio_seconds": ":.2f",
        "summary_pca_1": ":.3f",
        "summary_pca_2": ":.3f",
    },
    title="Summarized PCA of RAVE latent sequences",
    width=950,
    height=700,
)

fig_summary.show()

# zoomed plot
plot_summary_df = metadata.copy()

if "file_label" not in plot_summary_df.columns:
    plot_summary_df["file_label"] = plot_summary_df["file"].astype(str).apply(lambda x: Path(x).name)

fig_summary = px.scatter(
    plot_summary_df,
    x="summary_pca_1",
    y="summary_pca_2",
    color="source_type",
    hover_name="file_label",
    hover_data={
        "file": True,
        "source_type": True,
        "audio_seconds": ":.2f",
        "summary_pca_1": ":.3f",
        "summary_pca_2": ":.3f",
    },
    title="Summarized PCA of RAVE latent sequences",
    width=950,
    height=700,
)

fig_summary.update_xaxes(range=[-5, 5])
fig_summary.update_yaxes(range=[-5, 5])

fig_summary.show()

In the summarized plot, the clustering of birds is more on the left, songs more on the right side. But there is still some kind of overlap.

For the music generation step, we would want to select bird sounds that are more centrally located in the latent space and closer to the music samples, rather than using outliers. This should make the latent wandering smoother and lead to more coherent generated audio.

## 8. Alternative PCA component views

TODO

## 9. Generate music by walking between points in latent space

### Approach
   1. Build music_anchors from MANUAL_ANCHOR_FILES.
   2. Interpolate between neighbouring anchors
   3. Decode the intermediate latent points into audio
   4. Visualize anchors and intermediate points in PCA latent space
   5. Crossfade, save, and listen


### For example
6 anchors → 5 anchor gaps<br>
5 intermediate points per gap<br>
5 × 5 = 25 generated sections<br>

t_1 = 0.1 →  0–4 s<br>
t_2 = 0.3 →  4–8 s<br>
t_3 = 0.5 →  8–12 s<br>
t_4 = 0.7 → 12–16 s<br>
t_5 = 0.9 → 16–20 s<br>

The chosen anchors themselves aren't used. t_1 and t_5 gives us points close to the anchors, but they are still not the same as the original. This is done to have more focus on the generation part.

In [ ]:
# Settings
MANUAL_ANCHOR_FILES = [

    # first 4 snippets = first 1min20 (1min10 after crossfade): Nightjar Original Redefined 
    # 4 gaps × 5 generated points = 20 small segmente -> 19 crossfades -> 9.5 s shorter
    "Nightjar [6bdv61ud4Dw]_off0s_part1", 
    "Nightjar [6bdv61ud4Dw]_off0s_part2",
    "Nightjar [6bdv61ud4Dw]_off0s_part3",
    "Nightjar [6bdv61ud4Dw]_off0s_part4",

    # next 4 snippets: similar songs
    "Skylark [h8eCnWe4JRE]_off0s_part4.wav",
    "Nightingale, Pt. 2 [yLRgz96wZj0]_off0s_part3.wav",
    "Mistle Thrush [LcsiSs2p_sk]_off0s_part2.wav",
    "Dunnock [mZma45Dh-LA]_off0s_part4.wav",

    # back to nightjar for 20 seconds
    "Nightjar [6bdv61ud4Dw]_off10s_part4",               
    "Nightjar [6bdv61ud4Dw]_off10s_part5",               

]


# Number of generated latent points between each pair of anchors
# With 20-second decoded snippets and 4-second sections, 5 points are ideal:
# 20 s / 4 s = 5 windows
NUM_INTERMEDIATE_STEPS = 5

# Length of each decoded section used in the final audio
SEGMENT_SEC = 4.0

# Interpolation range:
# t = 0.0 would be anchor A.
# t = 1.0 would be anchor B.
# We avoid 0.0 and 1.0 because we want intermediate points, not the original anchors
T_START = 0.1
T_END = 0.9

t_values = np.round(
    np.linspace(T_START, T_END, NUM_INTERMEDIATE_STEPS),
    2
)

# Crossfade between final decoded sections
CROSSFADE_SEC = 0.5

print(f"  Manual anchors             : {len(MANUAL_ANCHOR_FILES)}")
print(f"  Intermediate steps per gap : {NUM_INTERMEDIATE_STEPS}")
print(f"  Segment length             : {SEGMENT_SEC:.1f}s")
print(f"  Interpolation t values     : {t_values}")
print(f"  Crossfade                  : {CROSSFADE_SEC:.1f}s")


In [ ]:
# Build music_anchors from MANUAL_ANCHOR_FILES
ANCHOR_SOURCES = [
    "song_training_nightjar",
    "song_training_similar",
]

music_anchors = [
    metadata[
        metadata["source_type"].isin(ANCHOR_SOURCES)
        & metadata["file"].astype(str).str.contains(
            pattern,
            case=False,
            regex=False,
        )
    ].index[0]
    for pattern in MANUAL_ANCHOR_FILES
]

print(f"\nBuilt music_anchors from {len(MANUAL_ANCHOR_FILES)} manual files:")

for i, idx in enumerate(music_anchors):
    file_name = Path(metadata.loc[idx, "file"]).name
    source_type = metadata.loc[idx, "source_type"]
    print(f"  anchor {i}: metadata index {idx} → {file_name} ({source_type})")

In [ ]:
# Project original anchors into summarized PCA space
# This is only for visualization.
anchor_pca_coords = []

for idx in music_anchors:
    z = z_list[idx]

    summary_vec = latent_summary(z).reshape(1, -1)
    pca_2d = pca_summary.transform(summary_vec)[0]

    anchor_pca_coords.append(pca_2d)

anchor_pca_coords = np.array(anchor_pca_coords)

anchor_df = pd.DataFrame({
    "anchor_id": list(range(len(music_anchors))),
    "metadata_index": music_anchors,
    "file_label": [Path(metadata.loc[idx, "file"]).name for idx in music_anchors],
    "summary_pca_1": anchor_pca_coords[:, 0],
    "summary_pca_2": anchor_pca_coords[:, 1],
    "source_type": "manual_music_anchor",
})

In [ ]:
# Build intermediate latent points between neighbouring anchors
#
# For each pair:
#
#   anchor A ---- intermediate points ---- anchor B
#
# Normally we generate 5 intermediate points:
#
#   point 1 -> window 0 ->  0–4 s
#   point 2 -> window 1 ->  4–8 s
#   point 3 -> window 2 ->  8–12 s
#   point 4 -> window 3 -> 12–16 s
#   point 5 -> window 4 -> 16–20 s
#
# If an anchor pair is shorter than 20 seconds, we generate fewer points
# so that every point still has a valid 4-second window.


latent_path = []
path_rows = []

max_steps_per_gap = NUM_INTERMEDIATE_STEPS

for gap_idx in range(len(music_anchors) - 1):

    idx_a = music_anchors[gap_idx]
    idx_b = music_anchors[gap_idx + 1]

    z_a = z_list[idx_a]
    z_b = z_list[idx_b]
    
    # Trim both anchors to the same latent length
    T = min(z_a.shape[-1], z_b.shape[-1])
    z_a = z_a[..., :T]
    z_b = z_b[..., :T]
    usable_sec = min(
        float(metadata.loc[idx_a, "audio_seconds"]),
        float(metadata.loc[idx_b, "audio_seconds"]),
    )

    # Number of full SEGMENT_SEC windows available
    steps_this_gap = min(
        NUM_INTERMEDIATE_STEPS,
        int(usable_sec // SEGMENT_SEC),
    )

    t_values_this_gap = np.round(
        np.linspace(T_START, T_END, steps_this_gap),
        2,
    )

    print(
        f"Gap {gap_idx}: "
        f"usable={usable_sec:.2f}s, "
        f"{steps_this_gap} points"
    )

    for point_inside_gap, t in enumerate(t_values_this_gap):

        z_interp = (1.0 - t) * z_a + t * z_b

        latent_path.append(z_interp)

        pca_2d = pca_summary.transform(
            latent_summary(z_interp).reshape(1, -1)
        )[0]

        path_rows.append({
            "path_index": len(latent_path) - 1,
            "point_type": "intermediate",
            "gap_index": gap_idx,
            "point_inside_gap": point_inside_gap,
            "window_index": point_inside_gap,
            "t": float(t),
            "from_anchor": gap_idx,
            "to_anchor": gap_idx + 1,
            "usable_sec": usable_sec,
            "steps_this_gap": steps_this_gap,
            "summary_pca_1": pca_2d[0],
            "summary_pca_2": pca_2d[1],
        })

path_df = pd.DataFrame(path_rows)

if len(path_df) == 0:
    raise RuntimeError(
        "No latent path points were created. "
        "Check music_anchors, audio_seconds, SEGMENT_SEC, and NUM_INTERMEDIATE_STEPS."
    )

n_intermediate = (path_df["point_type"] == "intermediate").sum()

print("\nLatent path built.")
print(f"  Manual anchors used                : {len(music_anchors)}")
print(f"  Anchor gaps checked                : {len(music_anchors) - 1}")
print(f"  Generated intermediate points      : {n_intermediate}")
print(f"  Total decoded latent points        : {len(latent_path)}")
print(f"  Estimated duration before crossfade: {len(latent_path) * SEGMENT_SEC:.1f} s")

display(path_df)

In [ ]:
# Visualize anchors and intermediate points in PCA latent space

plot_df = metadata.copy()
plot_df["file_label"] = plot_df["file"].astype(str).apply(lambda x: Path(x).name)

# Background groups for comparison:
# bird training data, similar training data, and Nightjar training data
wanted_sources = [
    source for source in metadata["source_type"].dropna().unique()
    if (
        "bird" in source.lower()
        or "similar" in source.lower()
        or "nightjar" in source.lower()
    )
]

plot_df = plot_df[plot_df["source_type"].isin(wanted_sources)].copy()

# define colours
color_map = {
    "song_training_bird": "#4C78A8",      # blue
    "song_training_similar": "#72B7B2",   # teal
    "song_training_nightjar": "#59A14F",  # green
}

fig_path = px.scatter(
    plot_df,
    x="summary_pca_1",
    y="summary_pca_2",
    color="source_type",
    color_discrete_map=color_map,
    hover_name="file_label",
    hover_data={
        "file": True,
        "source_type": True,
        "audio_seconds": ":.2f",
        "summary_pca_1": ":.3f",
        "summary_pca_2": ":.3f",
    },
    title="Intermediate latent points between manual Nightjar music anchors",
    width=950,
    height=700,
)

# Make background points smaller and transparent
for trace in fig_path.data:
    trace.update(
        marker=dict(
            size=6,
            opacity=0.35,
            line=dict(width=0),
        )
    )

# Dashed line through the original anchors
fig_path.add_trace(
    go.Scatter(
        x=anchor_df["summary_pca_1"],
        y=anchor_df["summary_pca_2"],
        mode="lines",
        line=dict(
            dash="dash",
            width=2,
            color="rgba(70, 70, 70, 0.65)",
        ),
        name="Anchor path",
        hoverinfo="skip",
    )
)

# Generated intermediate points, coloured by anchor gap
intermediate_df = path_df[path_df["point_type"] == "intermediate"].copy()

# One colour per gap / anchor pair
gap_colors = px.colors.qualitative.Safe + px.colors.qualitative.Set2 + px.colors.qualitative.Dark24

def color_to_rgba(color, alpha=0.18):
    """Convert Plotly color strings like '#RRGGBB' or 'rgb(r,g,b)' to 'rgba(r,g,b,alpha)'."""
    color = str(color).strip()

    if color.startswith("#"):
        hex_color = color.lstrip("#")
        r = int(hex_color[0:2], 16)
        g = int(hex_color[2:4], 16)
        b = int(hex_color[4:6], 16)
        return f"rgba({r},{g},{b},{alpha})"

    if color.startswith("rgb("):
        nums = color.replace("rgb(", "").replace(")", "").split(",")
        r, g, b = [int(float(n.strip())) for n in nums[:3]]
        return f"rgba({r},{g},{b},{alpha})"

    if color.startswith("rgba("):
        nums = color.replace("rgba(", "").replace(")", "").split(",")
        r, g, b = [int(float(n.strip())) for n in nums[:3]]
        return f"rgba({r},{g},{b},{alpha})"

    # fallback: light gray transparent line
    return f"rgba(80,80,80,{alpha})"


# Full connector line across all generated intermediate points
# This connects: last point of gap 0 -> first point of gap 1 -> ...
full_intermediate_path = intermediate_df.sort_values("path_index").copy()

fig_path.add_trace(
    go.Scatter(
        x=full_intermediate_path["summary_pca_1"],
        y=full_intermediate_path["summary_pca_2"],
        mode="lines",
        line=dict(
            width=2,
            color="rgba(40, 40, 40, 0.35)",
        ),
        name="Full generated path connector",
        hoverinfo="skip",
        showlegend=True,
    )
)

for gap_idx, gap_df in intermediate_df.groupby("gap_index"):

    gap_df = gap_df.sort_values("point_inside_gap").copy()

    from_anchor = gap_df["from_anchor"].iloc[0]
    to_anchor = gap_df["to_anchor"].iloc[0]

    marker_color = gap_colors[int(gap_idx) % len(gap_colors)]
    line_color = color_to_rgba(marker_color, alpha=0.2)

    fig_path.add_trace(
        go.Scatter(
            x=gap_df["summary_pca_1"],
            y=gap_df["summary_pca_2"],
            mode="markers+lines+text",
            marker=dict(
                size=8,
                color=marker_color,
                opacity=0.95,
                line=dict(
                    width=1,
                    color="black",
                ),
            ),
            line=dict(
                width=2,
                color=line_color,
            ),
            text=gap_df["point_inside_gap"].astype(str),
            textposition="top center",
            name=f"Generated: anchor {from_anchor} → {to_anchor}",
            customdata=np.stack(
                [
                    gap_df["path_index"],
                    gap_df["gap_index"],
                    gap_df["from_anchor"],
                    gap_df["to_anchor"],
                    gap_df["t"],
                    gap_df["window_index"],
                    gap_df["point_inside_gap"],
                ],
                axis=-1,
            ),
            hovertemplate=(
                "<b>Generated intermediate point</b><br>"
                "Path index: %{customdata[0]}<br>"
                "Gap index: %{customdata[1]}<br>"
                "From anchor: %{customdata[2]}<br>"
                "To anchor: %{customdata[3]}<br>"
                "t: %{customdata[4]:.2f}<br>"
                "Window index: %{customdata[5]}<br>"
                "Point inside gap: %{customdata[6]}<br>"
                "summary_pca_1: %{x:.3f}<br>"
                "summary_pca_2: %{y:.3f}<extra></extra>"
            ),
        )
    )

# Original anchor points
fig_path.add_trace(
    go.Scatter(
        x=anchor_df["summary_pca_1"],
        y=anchor_df["summary_pca_2"],
        mode="markers+text",
        marker=dict(
            symbol="x",
            size=5,
            color="#D62728",  # red
            opacity=1.0,
            line=dict(
                width=2,
                color="#7A0000",
            ),
        ),
        text=anchor_df["anchor_id"].astype(str),
        textposition="top right",
        textfont=dict(
            size=12,
            color="#D62728",
        ),
        name=f"Manual anchors ({len(anchor_df)})",
        customdata=np.stack(
            [
                anchor_df["anchor_id"],
                anchor_df["metadata_index"],
                anchor_df["file_label"],
            ],
            axis=-1,
        ),
        hovertemplate=(
            "<b>Anchor %{customdata[0]}</b><br>"
            "Metadata index: %{customdata[1]}<br>"
            "File: %{customdata[2]}<br>"
            "summary_pca_1: %{x:.3f}<br>"
            "summary_pca_2: %{y:.3f}<extra></extra>"
        ),
    )
)

# Match the previous summarized PCA view
fig_path.update_xaxes(
    range=[-2.5, 0.5],
    zeroline=False,
    showgrid=True,
    gridcolor="rgba(0,0,0,0.08)",
)

fig_path.update_yaxes(
    range=[-1.5, 0.5],
    zeroline=False,
    showgrid=True,
    gridcolor="rgba(0,0,0,0.08)",
)

fig_path.update_layout(
    plot_bgcolor="white",
    paper_bgcolor="white",
    legend=dict(
        title="Type",
        bgcolor="rgba(255,255,255,0.85)",
        bordercolor="rgba(0,0,0,0.15)",
        borderwidth=1,
    ),
)

fig_path.show()

Here in the plot, it may look like the generated points aren't close to the coresponding achors. Note here that this plot shows not the original latent space, but the latent space that we get when we summarize all points for one training snippet into one new point. The intermediate points are close to the original latent space points, but may not be close in the summary plot.

In [ ]:
# Decode generated latent points into audio segments
#
# We take sequential 4-second windows according to window_index:
#
#   window_index = 0 ->  0–4 s
#   window_index = 1 ->  4–8 s
#   window_index = 2 ->  8–12 s
#   window_index = 3 -> 12–16 s
#   window_index = 4 -> 16–20 s
#
# If a gap has fewer intermediate points because the source is shorter,
# it simply uses fewer windows. For example:
#   2 points -> windows 0 and 1 -> 0–4 s and 4–8 s

target_samples = int(SEGMENT_SEC * SR)
crossfade_samples = int(CROSSFADE_SEC * SR)

song_segments = []
segment_window_rows = []


def format_time(seconds: float) -> str:
    """Format seconds as M:SS or H:MM:SS if needed."""
    seconds = int(round(seconds))
    h = seconds // 3600
    m = (seconds % 3600) // 60
    s = seconds % 60

    if h > 0:
        return f"{h}:{m:02d}:{s:02d}"
    return f"{m}:{s:02d}"


for i, z in enumerate(tqdm(latent_path, desc="Decoding generated latent points")):

    y_full = rave_decode(z)

    row = path_df.iloc[i]

    window_idx = int(row["window_index"])
    gap_idx = int(row["gap_index"])
    t_value = float(row["t"])

    start_sample = window_idx * target_samples
    end_sample = start_sample + target_samples

    if end_sample > len(y_full):
        raise RuntimeError(
            f"Decoded audio is too short for this window.\n"
            f"Path index: {i}\n"
            f"Gap index: {gap_idx}\n"
            f"t value: {t_value}\n"
            f"Requested window: {start_sample / SR:.1f}–{end_sample / SR:.1f} s\n"
            f"Decoded duration: {len(y_full) / SR:.1f} s\n\n"
            f"Possible fixes:\n"
            f"  - reduce NUM_INTERMEDIATE_STEPS\n"
            f"  - reduce SEGMENT_SEC\n"
            f"  - use longer anchor snippets\n"
        )

    y_segment = y_full[start_sample:end_sample]

    # Position before crossfade
    before_crossfade_start_sec = i * SEGMENT_SEC
    before_crossfade_end_sec = before_crossfade_start_sec + SEGMENT_SEC
    time_before_crossfade = (
        f"{format_time(before_crossfade_start_sec)}–"
        f"{format_time(before_crossfade_end_sec)}"
    )

    # Estimated position after crossfade in the final assembled song
    # Each previous transition removes CROSSFADE_SEC from the timeline.
    final_start_sec = before_crossfade_start_sec - (i * CROSSFADE_SEC)
    final_end_sec = final_start_sec + SEGMENT_SEC
    final_time = (
        f"{format_time(final_start_sec)}–"
        f"{format_time(final_end_sec)}"
    )

    segment_window_rows.append({
        #"path_index": i,
        "gap_index": gap_idx,
        "t": t_value,
        "window_index": window_idx, 

        # Window taken from the decoded full audio for this latent point
        "source_start_sec": start_sample / SR,
        "source_end_sec": end_sample / SR,

        # Position before crossfade
        # "before_crossfade_start_sec": before_crossfade_start_sec,
        # "before_crossfade_end_sec": before_crossfade_end_sec,
        # "time_before_crossfade": time_before_crossfade,

        # Estimated position in the final generated audio after crossfades
        "final_time": final_time,
        "final_start_sec": final_start_sec,
        "final_end_sec": final_end_sec,
    })

    song_segments.append(y_segment)

segment_window_df = pd.DataFrame(segment_window_rows)

print(f"\nDecoded {len(song_segments)} audio segments.")
print(f"  Segment length before crossfade : {SEGMENT_SEC:.1f} s")
print(f"  Total before crossfade          : {len(song_segments) * SEGMENT_SEC:.1f} s")

display(segment_window_df)

In [ ]:
# Join decoded segments with equal-power crossfade
song = song_segments[0].copy()

for segment in song_segments[1:]:

    fade = min(crossfade_samples, len(song), len(segment))

    t = np.linspace(0, 1, fade)

    fade_out = np.cos(t * np.pi / 2)
    fade_in = np.sin(t * np.pi / 2)

    overlap = song[-fade:] * fade_out + segment[:fade] * fade_in

    song = np.concatenate([
        song[:-fade],
        overlap,
        segment[fade:],
    ])

print("\nFinal song assembled.")

In [ ]:
out_wav = OUTPUT_DIR / "generated_song.wav"

sf.write(str(out_wav), song, SR)

print("\nSaved:", out_wav)

display(Markdown("## Generated song"))
display(Audio(song, rate=SR))

## 10. Generate music by walking between points in latent space and then moving "into the wild"

### Approach
Same as above, but walk "into the wild" after the last musical anchor. This is done by moving to the center of the latent space first, then walking to the outer latent space.


- We define `PRE_WILD_ANCHOR_FILES` and `POST_WILD_ANCHOR_FILES`. The generated path first follows the pre-wild anchors. Then, from the last anchor in `PRE_WILD_ANCHOR_FILES`, it goes into the wild. After the wild detour, it returns to the first anchor in `POST_WILD_ANCHOR_FILES`.

- The wild detour is constructed in the **direct RAVE latent PCA space**, not in the summary PCA space. The direct PCA is based on the actual RAVE latent time steps.

- Each anchor is originally not one point, but a full latent sequence with many time steps. To get one representative position for the last pre-wild anchor, all its latent time steps are projected into the direct PCA space and then averaged. This mean point is used to decide the outward direction of the wild curve.

- The audio generation still uses the full latent tensor, not only this mean point. The mean only helps define the direction in the 2D PCA plot. This direction is converted back into an 8-dimensional latent offset and applied to the full latent sequence, so all latent time steps move together.

- `WILDE_LATENT_PCA_RADIUS` controls how far the wild target is pushed away in the direct PCA space. The `latent_pca_radius` stored for each generated wild point measures how far that point is from the origin of the direct PCA plot.

- The curve shape is created by adding a small perpendicular offset, controlled by `WILDE_CURVE_AMOUNT`. This makes the wild path bend instead of moving in a straight line.

- The summary PCA is used separately for visualization. After the wild points are generated, each full latent tensor is summarized with `latent_summary(z)` and projected into summary PCA space. This gives an overview of where the generated points sit compared to the broader training data and anchors.

In [ ]:
# Settings

PRE_WILD_ANCHOR_FILES  = [
    # first 4 snippets = first 1min20 (1min10 after crossfade): Nightjar Original Redefined 
    # 4 gaps × 5 generated points = 20 small segmente -> 19 crossfades -> 9.5 s shorter
    "songs/Nightjar [6bdv61ud4Dw]_off0s_part1", 
    "songs/Nightjar [6bdv61ud4Dw]_off0s_part2",
    "songs/Nightjar [6bdv61ud4Dw]_off0s_part3",
    "songs/Nightjar [6bdv61ud4Dw]_off0s_part4",

    # next 4 snippets: similar songs
    "songs/Skylark [h8eCnWe4JRE]_off0s_part4.wav",
    "songs/Nightingale, Pt. 2 [yLRgz96wZj0]_off0s_part3.wav",
    "songs/Mistle Thrush [LcsiSs2p_sk]_off0s_part2.wav",
    "songs/Dunnock [mZma45Dh-LA]_off0s_part4.wav",

    # back to nightjar before wild
    "songs/Nightjar [6bdv61ud4Dw]_off10s_part4",               
    "songs/Nightjar [6bdv61ud4Dw]_off10s_part5",   
]

# then: into the wild

# Fixed anchor where the post-wild section starts.
# The 3 bird anchors after this will be selected automatically.
POST_WILD_START_ANCHOR_FILE = "songs/Nightjar [6bdv61ud4Dw]_off10s_part5"
NUM_AUTO_BIRD_ANCHORS = 3

# Into-wild settings
WILDE_LATENT_PCA_RADIUS = 1.5
WILDE_STEPS = 10
WILDE_RETURN_STEPS = 10
WILDE_CURVE_AMOUNT = 0.3
WILDE_DIRECTION = -1   # +1 = same outward direction, -1 = opposite direction
SEGMENT_SEC = 4.0      # Length of each decoded section used in the final audio

# Interpolation range:
# t = 0.0 would be anchor A.
# t = 1.0 would be anchor B.
# We avoid 0.0 and 1.0 because we want intermediate points, not the original anchors.
T_START = 0.1
T_END = 0.9

t_values = np.round(
    np.linspace(T_START, T_END, NUM_INTERMEDIATE_STEPS),
    2
)

# Crossfade between final decoded sections
CROSSFADE_SEC = 0.5


# Auto-select post-wild bird anchors

# Find fixed post-wild start anchor
pattern = POST_WILD_START_ANCHOR_FILE
pattern_name = Path(pattern).name

file_as_string = metadata["file"].astype(str)

start_matches = metadata[
    file_as_string.str.contains(pattern, case=False, regex=False)
    | file_as_string.str.contains(pattern_name, case=False, regex=False)
]

if len(start_matches) == 0:
    raise RuntimeError(f"No match found for post-wild start anchor: {pattern}")

post_wild_start_idx = start_matches.index[0]

# Candidate bird anchors
bird_candidates = metadata[
    metadata["source_type"].astype(str).str.lower() == "bird_training"
].copy()

if len(bird_candidates) == 0:
    raise RuntimeError("No bird candidates found with source_type == 'bird_training'.")

bird_candidates["bird_type"] = (
    bird_candidates["file"]
    .astype(str)
    .apply(lambda x: Path(x).name.split("_sound_")[0])
)

chosen_bird_indices = []
chosen_bird_types = []

# Start by comparing birds to the fixed Nightjar post-wild start anchor
current_vec = latent_summary(z_list[post_wild_start_idx]).reshape(-1)

for step in range(NUM_AUTO_BIRD_ANCHORS):

    candidates = bird_candidates.copy()

    # Do not choose the exact same file twice
    candidates = candidates[~candidates.index.isin(chosen_bird_indices)]

    # Avoid same bird type as the previous selected bird
    if len(chosen_bird_types) > 0:
        candidates = candidates[
            candidates["bird_type"] != chosen_bird_types[-1]
        ]

    if len(candidates) == 0:
        raise RuntimeError(
            f"No candidate birds left at auto-bird step {step + 1}."
        )

    distances = []

    for idx in candidates.index:
        candidate_vec = latent_summary(z_list[idx]).reshape(-1)
        distance = np.linalg.norm(candidate_vec - current_vec) # Euclidean distance between two latent-summary vectors
        distances.append(distance)

    candidates = candidates.copy()
    candidates["distance"] = distances

    best_idx = candidates.sort_values("distance").index[0]
    best_bird_type = candidates.loc[best_idx, "bird_type"]

    chosen_bird_indices.append(best_idx)
    chosen_bird_types.append(best_bird_type)

    # Next bird is selected relative to this bird
    current_vec = latent_summary(z_list[best_idx]).reshape(-1)

    print(
        f"Auto bird {step + 1}: "
        f"{Path(metadata.loc[best_idx, 'file']).name} "
        f"({best_bird_type}), "
        f"distance={candidates.loc[best_idx, 'distance']:.4f}"
    )

# Final post-wild anchors:
# fixed Nightjar anchor, then automatically selected birds
post_wild_anchors = [post_wild_start_idx] + chosen_bird_indices

POST_WILD_ANCHOR_FILES = [
    metadata.loc[idx, "file"]
    for idx in post_wild_anchors
]

# Combined list only for plotting / anchor overview
MANUAL_ANCHOR_FILES = PRE_WILD_ANCHOR_FILES + POST_WILD_ANCHOR_FILES


print("\nPOST_WILD_ANCHOR_FILES:")
for file in POST_WILD_ANCHOR_FILES:
    print(f"  {file}")

print(f"\n  Pre-wild anchors          : {len(PRE_WILD_ANCHOR_FILES)}")
print(f"  Post-wild anchors         : {len(POST_WILD_ANCHOR_FILES)}")
print(f"  Total manual anchors      : {len(MANUAL_ANCHOR_FILES)}")
print(f"  Intermediate steps per gap: {NUM_INTERMEDIATE_STEPS}")
print(f"  Segment length            : {SEGMENT_SEC:.1f}s")
print(f"  Interpolation t values    : {t_values}")
print(f"  Crossfade                 : {CROSSFADE_SEC:.1f}s")

In [ ]:
# Build anchors from file names

ANCHOR_SOURCES = [
    "song_training_nightjar",
    "song_training_similar",
    "bird_training"
]

def find_anchor_indices(anchor_files, label):
    anchor_indices = []

    for pattern in anchor_files:
        matches = metadata[
            metadata["source_type"].isin(ANCHOR_SOURCES)
            & metadata["file"].astype(str).str.contains(
                pattern,
                case=False,
                regex=False,
            )
        ]

        if len(matches) == 0:
            raise RuntimeError(f"No match found for {label} anchor pattern: {pattern}")

        anchor_indices.append(matches.index[0])

    return anchor_indices


pre_wild_anchors = find_anchor_indices(PRE_WILD_ANCHOR_FILES, "pre-wild")
post_wild_anchors = find_anchor_indices(POST_WILD_ANCHOR_FILES, "post-wild")

# Combined list for plotting all manual anchors
music_anchors = pre_wild_anchors + post_wild_anchors

print("\nPre-wild anchors:")
for i, idx in enumerate(pre_wild_anchors):
    file_name = Path(metadata.loc[idx, "file"]).name
    source_type = metadata.loc[idx, "source_type"]
    print(f"  pre {i}: metadata index {idx} → {file_name} ({source_type})")

print("\nPost-wild anchors:")
for i, idx in enumerate(post_wild_anchors):
    file_name = Path(metadata.loc[idx, "file"]).name
    source_type = metadata.loc[idx, "source_type"]
    print(f"  post {i}: metadata index {idx} → {file_name} ({source_type})")

print(f"\nCombined manual anchors for plotting: {len(music_anchors)}")

In [ ]:
# Project original anchors into summarized PCA space
# This is only for visualization.

anchor_pca_coords = []

for idx in music_anchors:
    z = z_list[idx]

    summary_vec = latent_summary(z).reshape(1, -1)
    pca_2d = pca_summary.transform(summary_vec)[0]

    anchor_pca_coords.append(pca_2d)

anchor_pca_coords = np.array(anchor_pca_coords)

anchor_phase = (
    ["pre_wild"] * len(pre_wild_anchors)
    + ["post_wild"] * len(post_wild_anchors)
)

anchor_df = pd.DataFrame({
    "anchor_id": list(range(len(music_anchors))),
    "metadata_index": music_anchors,
    "anchor_phase": anchor_phase,
    "file_label": [Path(metadata.loc[idx, "file"]).name for idx in music_anchors],
    "summary_pca_1": anchor_pca_coords[:, 0],
    "summary_pca_2": anchor_pca_coords[:, 1],
    "source_type": "manual_music_anchor",
})

In [ ]:
# Build latent path:
#
#   pre-wild anchors
#   → walk into wild from last pre-wild anchor
#   → return from wild to first post-wild anchor
#   → continue through remaining post-wild anchors

latent_path = []
path_rows = []


def add_interpolation_gap(
    idx_a,
    idx_b,
    gap_idx,
    from_anchor_label,
    to_anchor_label,
    point_type="intermediate",
):
    """Add normal interpolation points between two anchors."""

    z_a = z_list[idx_a]
    z_b = z_list[idx_b]

    # Trim both anchors to the same latent length
    T = min(z_a.shape[-1], z_b.shape[-1])
    z_a = z_a[..., :T]
    z_b = z_b[..., :T]

    usable_sec = min(
        float(metadata.loc[idx_a, "audio_seconds"]),
        float(metadata.loc[idx_b, "audio_seconds"]),
    )

    steps_this_gap = min(
        NUM_INTERMEDIATE_STEPS,
        int(usable_sec // SEGMENT_SEC),
    )

    t_values_this_gap = np.round(
        np.linspace(T_START, T_END, steps_this_gap),
        2,
    )

    print(
        f"Gap {gap_idx}: "
        f"{from_anchor_label} → {to_anchor_label}, "
        f"usable={usable_sec:.2f}s, "
        f"{steps_this_gap} points"
    )

    for point_inside_gap, t in enumerate(t_values_this_gap):

        z_interp = (1.0 - t) * z_a + t * z_b

        latent_path.append(z_interp)

        pca_2d = pca_summary.transform(
            latent_summary(z_interp).reshape(1, -1)
        )[0]

        path_rows.append({
            "path_index": len(latent_path) - 1,
            "point_type": point_type,
            "gap_index": gap_idx,
            "point_inside_gap": point_inside_gap,
            "window_index": point_inside_gap,
            "t": float(t),
            "from_anchor": from_anchor_label,
            "to_anchor": to_anchor_label,
            "usable_sec": usable_sec,
            "steps_this_gap": steps_this_gap,
            "summary_pca_1": pca_2d[0],
            "summary_pca_2": pca_2d[1],
        })


# Normal path through pre-wild anchors only
gap_idx = 0

for local_gap_idx in range(len(pre_wild_anchors) - 1):

    idx_a = pre_wild_anchors[local_gap_idx]
    idx_b = pre_wild_anchors[local_gap_idx + 1]

    add_interpolation_gap(
        idx_a=idx_a,
        idx_b=idx_b,
        gap_idx=gap_idx,
        from_anchor_label=f"pre_{local_gap_idx}",
        to_anchor_label=f"pre_{local_gap_idx + 1}",
        point_type="intermediate",
    )

    gap_idx += 1


# Into the wild:
#    start = last pre-wild anchor
#    return target = first post-wild anchor

if len(post_wild_anchors) == 0:
    raise RuntimeError("POST_WILD_ANCHOR_FILES must contain at least one anchor.")

wild_start_idx = pre_wild_anchors[-1]
wild_return_idx = post_wild_anchors[0]

z_start = z_list[wild_start_idx]
z_return_anchor = z_list[wild_return_idx]

# Trim start and return anchor to same latent length
T = min(z_start.shape[-1], z_return_anchor.shape[-1])
z_start = z_start[..., :T]
z_return_anchor = z_return_anchor[..., :T]

# Direct PCA position of wild start anchor
z_start_steps = z_start.T
start_pca_steps = pca_latent.transform(z_start_steps)
start_xy = start_pca_steps[:, :2].mean(axis=0)
start_radius = np.linalg.norm(start_xy)

if start_radius == 0:
    raise RuntimeError(
        "Wild start anchor is exactly at origin in direct latent PCA space. "
        "Cannot define outward direction."
    )

# Direction from origin through wild start anchor
direction_xy = start_xy / start_radius

# Perpendicular direction for curve
perpendicular_xy = np.array([
    -direction_xy[1],
    direction_xy[0],
])

# Wild target in direct latent PCA space
target_radius = start_radius + abs(WILDE_LATENT_PCA_RADIUS)
target_xy = WILDE_DIRECTION * direction_xy * target_radius

# Convert PCA move to latent-channel move
delta_xy = target_xy - start_xy
delta_latent = delta_xy @ pca_latent.components_[:2]

curve_xy = perpendicular_xy * WILDE_CURVE_AMOUNT
curve_latent = curve_xy @ pca_latent.components_[:2]

z_wilde_target = z_start + delta_latent[:, None]

print(
    f"\nWALK INTO WILDE enabled:"
    f"\n  Wild start anchor metadata index  : {wild_start_idx}"
    f"\n  Wild return anchor metadata index : {wild_return_idx}"
    f"\n  Start direct PCA position         : ({start_xy[0]:.3f}, {start_xy[1]:.3f})"
    f"\n  Start direct PCA radius           : {start_radius:.3f}"
    f"\n  Wild target direct PCA position   : ({target_xy[0]:.3f}, {target_xy[1]:.3f})"
    f"\n  Wild target direct PCA radius     : {target_radius:.3f}"
    f"\n  Wild outward points               : {WILDE_STEPS}"
    f"\n  Wild return points                : {WILDE_RETURN_STEPS}"
)

# ---------- go out to wilde ----------

wilde_t_values = np.linspace(
    1 / WILDE_STEPS,
    1.0,
    WILDE_STEPS,
)

for point_inside_wilde, wilde_t in enumerate(wilde_t_values):

    z_wilde = (
        (1.0 - wilde_t) * z_start
        + wilde_t * z_wilde_target
        + np.sin(np.pi * wilde_t) * curve_latent[:, None]
    )

    latent_path.append(z_wilde)

    pca_2d = pca_summary.transform(
        latent_summary(z_wilde).reshape(1, -1)
    )[0]

    z_wilde_steps = z_wilde.T
    wilde_xy = pca_latent.transform(z_wilde_steps)[:, :2].mean(axis=0)
    wilde_radius = np.linalg.norm(wilde_xy)

    path_rows.append({
        "path_index": len(latent_path) - 1,
        "point_type": "wilde_out",
        "gap_index": gap_idx,
        "point_inside_gap": point_inside_wilde,
        "window_index": point_inside_wilde % NUM_INTERMEDIATE_STEPS,
        "t": np.nan,
        "wilde_t": float(wilde_t),
        "from_anchor": "wild_start",
        "to_anchor": "wilde",
        "usable_sec": float(metadata.loc[wild_start_idx, "audio_seconds"]),
        "steps_this_gap": WILDE_STEPS,
        "latent_pca_1": float(wilde_xy[0]),
        "latent_pca_2": float(wilde_xy[1]),
        "latent_pca_radius": float(wilde_radius),
        "summary_pca_1": pca_2d[0],
        "summary_pca_2": pca_2d[1],
    })

gap_idx += 1


# ---------- return from wilde to FIRST POST-WILD ANCHOR ----------

return_t_values = np.linspace(
    1 / WILDE_RETURN_STEPS,
    1.0,
    WILDE_RETURN_STEPS,
)

for point_inside_return, return_t in enumerate(return_t_values):

    # This now returns to z_return_anchor, not z_start.
    z_return = (
        (1.0 - return_t) * z_wilde_target
        + return_t * z_return_anchor
        - np.sin(np.pi * return_t) * curve_latent[:, None]
    )

    latent_path.append(z_return)

    pca_2d = pca_summary.transform(
        latent_summary(z_return).reshape(1, -1)
    )[0]

    z_return_steps = z_return.T
    return_xy = pca_latent.transform(z_return_steps)[:, :2].mean(axis=0)
    return_radius = np.linalg.norm(return_xy)

    path_rows.append({
        "path_index": len(latent_path) - 1,
        "point_type": "wilde_return",
        "gap_index": gap_idx,
        "point_inside_gap": point_inside_return,
        "window_index": point_inside_return % NUM_INTERMEDIATE_STEPS,
        "t": np.nan,
        "wilde_t": float(return_t),
        "from_anchor": "wilde",
        "to_anchor": "post_0",
        "usable_sec": float(metadata.loc[wild_return_idx, "audio_seconds"]),
        "steps_this_gap": WILDE_RETURN_STEPS,
        "latent_pca_1": float(return_xy[0]),
        "latent_pca_2": float(return_xy[1]),
        "latent_pca_radius": float(return_radius),
        "summary_pca_1": pca_2d[0],
        "summary_pca_2": pca_2d[1],
    })

gap_idx += 1

# Continue through remaining post-wild anchors
for local_gap_idx in range(len(post_wild_anchors) - 1):

    idx_a = post_wild_anchors[local_gap_idx]
    idx_b = post_wild_anchors[local_gap_idx + 1]

    add_interpolation_gap(
        idx_a=idx_a,
        idx_b=idx_b,
        gap_idx=gap_idx,
        from_anchor_label=f"post_{local_gap_idx}",
        to_anchor_label=f"post_{local_gap_idx + 1}",
        point_type="post_wild_intermediate",
    )

    gap_idx += 1

# Final DataFrame
path_df = pd.DataFrame(path_rows)

if len(path_df) == 0:
    raise RuntimeError(
        "No latent path points were created. "
        "Check anchors, audio_seconds, SEGMENT_SEC, and NUM_INTERMEDIATE_STEPS."
    )

print("\nLatent path built.")
print(f"  Pre-wild anchors used             : {len(pre_wild_anchors)}")
print(f"  Post-wild anchors used            : {len(post_wild_anchors)}")
print(f"  Total decoded latent points       : {len(latent_path)}")
print(f"  Estimated duration before crossfade: {len(latent_path) * SEGMENT_SEC:.1f} s")

display(path_df)

In [ ]:
print("Data for 'wilde' points:")
path_df[45:65]

In [ ]:
# Visualize anchors and generated points in summarized PCA latent space
#
# This plot shows:
#   - background training points in summarized PCA space
#   - manual anchors in summarized PCA space
#   - normal generated intermediate points
#   - post-wild generated intermediate points
#   - wilde outward + return points
#   - one continuous wilde connector:
#       wild start anchor → wilde_out → wilde_return → wild return anchor

plot_df = metadata.copy()
plot_df["file_label"] = plot_df["file"].astype(str).apply(lambda x: Path(x).name)

# Background groups for comparison:
# bird training data, similar training data, and Nightjar training data
wanted_sources = [
    source for source in metadata["source_type"].dropna().unique()
    if (
        "bird" in source.lower()
        or "similar" in source.lower()
        or "nightjar" in source.lower()
    )
]

plot_df = plot_df[plot_df["source_type"].isin(wanted_sources)].copy()

# Define colours
color_map = {
    "song_training_bird": "#4C78A8",      # blue
    "song_training_similar": "#72B7B2",   # teal
    "song_training_nightjar": "#59A14F",  # green
    "bird_training": "#4C78A8",           # blue, in case birds use this label
}

fig_path = px.scatter(
    plot_df,
    x="summary_pca_1",
    y="summary_pca_2",
    color="source_type",
    color_discrete_map=color_map,
    hover_name="file_label",
    hover_data={
        "file": True,
        "source_type": True,
        "audio_seconds": ":.2f",
        "summary_pca_1": ":.3f",
        "summary_pca_2": ":.3f",
    },
    title="Generated latent path in summarized PCA space, including wilde detour",
    width=950,
    height=700,
)

# Make background points smaller and transparent
for trace in fig_path.data:
    trace.update(
        marker=dict(
            size=6,
            opacity=0.35,
            line=dict(width=0),
        )
    )


# Dashed line through all original manual anchors
fig_path.add_trace(
    go.Scatter(
        x=anchor_df["summary_pca_1"],
        y=anchor_df["summary_pca_2"],
        mode="lines",
        line=dict(
            dash="dash",
            width=2,
            color="rgba(70, 70, 70, 0.65)",
        ),
        name="Manual anchor path",
        hoverinfo="skip",
    )
)

# Split generated path rows
normal_df = path_df[
    path_df["point_type"].isin([
        "intermediate",
        "post_wild_intermediate",
    ])
].copy()

wild_out_df = path_df[
    path_df["point_type"] == "wilde_out"
].copy()

wild_return_df = path_df[
    path_df["point_type"] == "wilde_return"
].copy()


gap_colors = (
    px.colors.qualitative.Safe
    + px.colors.qualitative.Set2
    + px.colors.qualitative.Dark24
)


def color_to_rgba(color, alpha=0.18):
    """Convert Plotly color strings like '#RRGGBB' or 'rgb(r,g,b)' to 'rgba(r,g,b,alpha)'."""
    color = str(color).strip()

    if color.startswith("#"):
        hex_color = color.lstrip("#")
        r = int(hex_color[0:2], 16)
        g = int(hex_color[2:4], 16)
        b = int(hex_color[4:6], 16)
        return f"rgba({r},{g},{b},{alpha})"

    if color.startswith("rgb("):
        nums = color.replace("rgb(", "").replace(")", "").split(",")
        r, g, b = [int(float(n.strip())) for n in nums[:3]]
        return f"rgba({r},{g},{b},{alpha})"

    if color.startswith("rgba("):
        nums = color.replace("rgba(", "").replace(")", "").split(",")
        r, g, b = [int(float(n.strip())) for n in nums[:3]]
        return f"rgba({r},{g},{b},{alpha})"

    return f"rgba(80,80,80,{alpha})"


# Full connector line across normal + post-wild generated points
normal_path = normal_df.sort_values("path_index").copy()

if len(normal_path) > 0:
    fig_path.add_trace(
        go.Scatter(
            x=normal_path["summary_pca_1"],
            y=normal_path["summary_pca_2"],
            mode="lines",
            line=dict(
                width=2,
                color="rgba(40, 40, 40, 0.35)",
            ),
            name="Generated music path connector",
            hoverinfo="skip",
            showlegend=True,
        )
    )


# Normal + post-wild generated intermediate points
for gap_idx, gap_df in normal_df.groupby("gap_index"):

    gap_df = gap_df.sort_values("point_inside_gap").copy()

    from_anchor = gap_df["from_anchor"].iloc[0]
    to_anchor = gap_df["to_anchor"].iloc[0]
    point_type = gap_df["point_type"].iloc[0]

    marker_color = gap_colors[int(gap_idx) % len(gap_colors)]
    line_color = color_to_rgba(marker_color, alpha=0.2)

    if point_type == "post_wild_intermediate":
        trace_name = f"Post-wild: {from_anchor} → {to_anchor}"
    else:
        trace_name = f"Generated: {from_anchor} → {to_anchor}"

    fig_path.add_trace(
        go.Scatter(
            x=gap_df["summary_pca_1"],
            y=gap_df["summary_pca_2"],
            mode="markers+lines",
            marker=dict(
                size=8,
                color=marker_color,
                opacity=0.95,
                line=dict(width=1, color="black"),
            ),
            line=dict(
                width=2,
                color=line_color,
            ),
            name=trace_name,
            customdata=np.stack(
                [
                    gap_df["path_index"],
                    gap_df["gap_index"],
                    gap_df["from_anchor"],
                    gap_df["to_anchor"],
                    gap_df["window_index"],
                    gap_df["point_inside_gap"],
                    gap_df["point_type"],
                ],
                axis=-1,
            ),
            hovertemplate=(
                "<b>%{fullData.name}</b><br>"
                "Path index: %{customdata[0]}<br>"
                "Gap index: %{customdata[1]}<br>"
                "From anchor: %{customdata[2]}<br>"
                "To anchor: %{customdata[3]}<br>"
                "Window index: %{customdata[4]}<br>"
                "Point inside gap: %{customdata[5]}<br>"
                "Point type: %{customdata[6]}<br>"
                "summary_pca_1: %{x:.3f}<br>"
                "summary_pca_2: %{y:.3f}<extra></extra>"
            ),
        )
    )


# Full continuous wilde connector
# Connects:
#   wild start anchor
#   → wilde_out points
#   → wilde_return points
#   → wild return anchor

if len(wild_out_df) > 0 and len(wild_return_df) > 0:

    wild_out_df = wild_out_df.sort_values("path_index").copy()
    wild_return_df = wild_return_df.sort_values("path_index").copy()

    wild_start_idx = pre_wild_anchors[-1]
    wild_return_idx = post_wild_anchors[0]

    wild_start_anchor_match = anchor_df[
        anchor_df["metadata_index"] == wild_start_idx
    ]

    wild_return_anchor_match = anchor_df[
        anchor_df["metadata_index"] == wild_return_idx
    ]

    if len(wild_start_anchor_match) == 0:
        raise RuntimeError(
            f"Could not find wild start anchor in anchor_df. "
            f"metadata_index={wild_start_idx}"
        )

    if len(wild_return_anchor_match) == 0:
        raise RuntimeError(
            f"Could not find wild return anchor in anchor_df. "
            f"metadata_index={wild_return_idx}"
        )

    wild_start_anchor_row = wild_start_anchor_match.iloc[0]
    wild_return_anchor_row = wild_return_anchor_match.iloc[0]

    wild_connector_rows = []

    # Start anchor
    wild_connector_rows.append({
        "point_label": "wild_start_anchor",
        "summary_pca_1": wild_start_anchor_row["summary_pca_1"],
        "summary_pca_2": wild_start_anchor_row["summary_pca_2"],
    })

    # Outward wild points
    for _, row in wild_out_df.iterrows():
        wild_connector_rows.append({
            "point_label": "wilde_out",
            "summary_pca_1": row["summary_pca_1"],
            "summary_pca_2": row["summary_pca_2"],
        })

    # Return wild points
    for _, row in wild_return_df.iterrows():
        wild_connector_rows.append({
            "point_label": "wilde_return",
            "summary_pca_1": row["summary_pca_1"],
            "summary_pca_2": row["summary_pca_2"],
        })

    # Return target anchor
    wild_connector_rows.append({
        "point_label": "wild_return_anchor",
        "summary_pca_1": wild_return_anchor_row["summary_pca_1"],
        "summary_pca_2": wild_return_anchor_row["summary_pca_2"],
    })

    wild_connector_df = pd.DataFrame(wild_connector_rows)

    fig_path.add_trace(
        go.Scatter(
            x=wild_connector_df["summary_pca_1"],
            y=wild_connector_df["summary_pca_2"],
            mode="lines",
            line=dict(
                width=8,
                color="rgba(160, 0, 0, 0.35)",
            ),
            name="Full wilde connector",
            hoverinfo="skip",
        )
    )


# Wilde outward path in summarized PCA space
if len(wild_out_df) > 0:

    wild_out_df = wild_out_df.sort_values("path_index").copy()

    fig_path.add_trace(
        go.Scatter(
            x=wild_out_df["summary_pca_1"],
            y=wild_out_df["summary_pca_2"],
            mode="lines",
            line=dict(
                width=5,
                color="red",
            ),
            name="Wilde outward, summarized",
            hoverinfo="skip",
        )
    )

    fig_path.add_trace(
        go.Scatter(
            x=wild_out_df["summary_pca_1"],
            y=wild_out_df["summary_pca_2"],
            mode="markers",
            marker=dict(
                size=9,
                color="white",
                symbol="circle",
                line=dict(width=3, color="red"),
            ),
            name="Wilde outward points",
            showlegend=False,
            customdata=np.stack(
                [
                    wild_out_df["path_index"],
                    wild_out_df["wilde_t"],
                    wild_out_df["window_index"],
                    wild_out_df["latent_pca_radius"],
                ],
                axis=-1,
            ),
            hovertemplate=(
                "<b>Wilde outward, summarized</b><br>"
                "Path index: %{customdata[0]}<br>"
                "Wilde t: %{customdata[1]:.2f}<br>"
                "Window index: %{customdata[2]}<br>"
                "Direct latent PCA radius: %{customdata[3]:.3f}<br>"
                "summary_pca_1: %{x:.3f}<br>"
                "summary_pca_2: %{y:.3f}<extra></extra>"
            ),
        )
    )


# Wilde return path in summarized PCA space
if len(wild_return_df) > 0:

    wild_return_df = wild_return_df.sort_values("path_index").copy()

    fig_path.add_trace(
        go.Scatter(
            x=wild_return_df["summary_pca_1"],
            y=wild_return_df["summary_pca_2"],
            mode="lines",
            line=dict(
                width=5,
                color="orange",
                dash="dash",
            ),
            name="Return from wilde, summarized",
            hoverinfo="skip",
        )
    )

    fig_path.add_trace(
        go.Scatter(
            x=wild_return_df["summary_pca_1"],
            y=wild_return_df["summary_pca_2"],
            mode="markers",
            marker=dict(
                size=9,
                color="white",
                symbol="diamond",
                line=dict(width=3, color="orange"),
            ),
            name="Return from wilde points",
            showlegend=False,
            customdata=np.stack(
                [
                    wild_return_df["path_index"],
                    wild_return_df["wilde_t"],
                    wild_return_df["window_index"],
                    wild_return_df["latent_pca_radius"],
                ],
                axis=-1,
            ),
            hovertemplate=(
                "<b>Return from wilde, summarized</b><br>"
                "Path index: %{customdata[0]}<br>"
                "Wilde t: %{customdata[1]:.2f}<br>"
                "Window index: %{customdata[2]}<br>"
                "Direct latent PCA radius: %{customdata[3]:.3f}<br>"
                "summary_pca_1: %{x:.3f}<br>"
                "summary_pca_2: %{y:.3f}<extra></extra>"
            ),
        )
    )

# Original manual anchor points

fig_path.add_trace(
    go.Scatter(
        x=anchor_df["summary_pca_1"],
        y=anchor_df["summary_pca_2"],
        mode="markers+text",
        marker=dict(
            symbol="x",
            size=6,
            color="#D62728",
            opacity=1.0,
            line=dict(
                width=2,
                color="#7A0000",
            ),
        ),
        text=anchor_df["anchor_id"].astype(str),
        textposition="top right",
        textfont=dict(
            size=12,
            color="#D62728",
        ),
        name=f"Manual anchors ({len(anchor_df)})",
        customdata=np.stack(
            [
                anchor_df["anchor_id"],
                anchor_df["metadata_index"],
                anchor_df["file_label"],
                anchor_df["anchor_phase"] if "anchor_phase" in anchor_df.columns else anchor_df["source_type"],
            ],
            axis=-1,
        ),
        hovertemplate=(
            "<b>Anchor %{customdata[0]}</b><br>"
            "Metadata index: %{customdata[1]}<br>"
            "File: %{customdata[2]}<br>"
            "Phase/source: %{customdata[3]}<br>"
            "summary_pca_1: %{x:.3f}<br>"
            "summary_pca_2: %{y:.3f}<extra></extra>"
        ),
    )
)

# Axes and layout
fig_path.update_xaxes(
    range=[-2.5, 2.5],
    zeroline=False,
    showgrid=True,
    gridcolor="rgba(0,0,0,0.08)",
)

fig_path.update_yaxes(
    range=[-3, 1.5],
    zeroline=False,
    showgrid=True,
    gridcolor="rgba(0,0,0,0.08)",
)

fig_path.update_layout(
    plot_bgcolor="white",
    paper_bgcolor="white",
    legend=dict(
        title="Type",
        bgcolor="rgba(255,255,255,0.85)",
        bordercolor="rgba(0,0,0,0.15)",
        borderwidth=1,
    ),
)

fig_path.show()

In [ ]:
# Decode generated latent points into audio segments
#
# We take sequential 4-second windows according to window_index:
#
#   window_index = 0 ->  0–4 s
#   window_index = 1 ->  4–8 s
#   window_index = 2 ->  8–12 s
#   window_index = 3 -> 12–16 s
#   window_index = 4 -> 16–20 s
#
# If a gap has fewer intermediate points because the source is shorter,
# it simply uses fewer windows. For example:
#   2 points -> windows 0 and 1 -> 0–4 s and 4–8 s

target_samples = int(SEGMENT_SEC * SR)
crossfade_samples = int(CROSSFADE_SEC * SR)

song_segments = []
segment_window_rows = []


def format_time(seconds: float) -> str:
    """Format seconds as M:SS or H:MM:SS if needed."""
    seconds = int(round(seconds))
    h = seconds // 3600
    m = (seconds % 3600) // 60
    s = seconds % 60

    if h > 0:
        return f"{h}:{m:02d}:{s:02d}"
    return f"{m}:{s:02d}"


for i, z in enumerate(tqdm(latent_path, desc="Decoding generated latent points")):

    y_full = rave_decode(z)

    row = path_df.iloc[i]

    window_idx = int(row["window_index"])
    gap_idx = int(row["gap_index"])
    t_value = float(row["t"])

    start_sample = window_idx * target_samples
    end_sample = start_sample + target_samples

    if end_sample > len(y_full):
        raise RuntimeError(
            f"Decoded audio is too short for this window.\n"
            f"Path index: {i}\n"
            f"Gap index: {gap_idx}\n"
            f"t value: {t_value}\n"
            f"Requested window: {start_sample / SR:.1f}–{end_sample / SR:.1f} s\n"
            f"Decoded duration: {len(y_full) / SR:.1f} s\n\n"
            f"Possible fixes:\n"
            f"  - reduce NUM_INTERMEDIATE_STEPS\n"
            f"  - reduce SEGMENT_SEC\n"
            f"  - use longer anchor snippets\n"
        )

    y_segment = y_full[start_sample:end_sample]

    # Position before crossfade
    before_crossfade_start_sec = i * SEGMENT_SEC
    before_crossfade_end_sec = before_crossfade_start_sec + SEGMENT_SEC
    time_before_crossfade = (
        f"{format_time(before_crossfade_start_sec)}–"
        f"{format_time(before_crossfade_end_sec)}"
    )

    # Estimated position after crossfade in the final assembled song
    # Each previous transition removes CROSSFADE_SEC from the timeline.
    final_start_sec = before_crossfade_start_sec - (i * CROSSFADE_SEC)
    final_end_sec = final_start_sec + SEGMENT_SEC
    final_time = (
        f"{format_time(final_start_sec)}–"
        f"{format_time(final_end_sec)}"
    )

    segment_window_rows.append({
        #"path_index": i,
        "gap_index": gap_idx,
        "t": t_value,
        "window_index": window_idx, 

        # Window taken from the decoded full audio for this latent point
        "source_start_sec": start_sample / SR,
        "source_end_sec": end_sample / SR,

        # Estimated position in the final generated audio after crossfades
        "final_time": final_time,
        "final_start_sec": final_start_sec,
        "final_end_sec": final_end_sec,
    })
 
    song_segments.append(y_segment)

segment_window_df = pd.DataFrame(segment_window_rows)

print(f"\nDecoded {len(song_segments)} audio segments.")
print(f"  Segment length before crossfade : {SEGMENT_SEC:.1f} s")
print(f"  Total before crossfade          : {len(song_segments) * SEGMENT_SEC:.1f} s")

display(segment_window_df)

In [ ]:
segment_window_df[30:]

In [ ]:
# Fast direct PCA plot for wilde walk
# Shows ALL latent_metadata points, but uses WebGL and lightweight hover.

plot_latent_df = latent_metadata.copy()

fig_latent = px.scatter(
    plot_latent_df,
    x="latent_pca_1",
    y="latent_pca_2",
    color="source_type",
    hover_data={
        "source_type": True,
        "latent_pca_1": ":.3f",
        "latent_pca_2": ":.3f",
    },
    title="Direct latent PCA: wilde detour",
    width=950,
    height=700,
    opacity=0.25,
    render_mode="webgl",
)

# Make all background traces lighter and smaller
for trace in fig_latent.data:
    trace.update(
        marker=dict(
            size=3,
            opacity=0.35,
        )
    )

fig_latent.update_xaxes(range=[-4, 4])
fig_latent.update_yaxes(range=[-4, 4])


# Anchor means only, no full anchor time-step clouds

wild_start_idx = pre_wild_anchors[-1]
wild_return_idx = post_wild_anchors[0]

z_start = z_list[wild_start_idx]
z_return_anchor = z_list[wild_return_idx]

start_pca_steps = pca_latent.transform(z_start.T)
return_pca_steps = pca_latent.transform(z_return_anchor.T)

start_xy = start_pca_steps[:, :2].mean(axis=0)
return_xy = return_pca_steps[:, :2].mean(axis=0)


# Wilde path rows
wilde_out_df = path_df[
    path_df["point_type"] == "wilde_out"
].sort_values("path_index").copy()

wilde_return_df = path_df[
    path_df["point_type"] == "wilde_return"
].sort_values("path_index").copy()

if len(wilde_out_df) == 0:
    raise RuntimeError("No wilde_out rows found in path_df.")

if len(wilde_return_df) == 0:
    raise RuntimeError("No wilde_return rows found in path_df.")


# Full connector:
# start anchor mean → wilde_out → wilde_return → return anchor mean
connector_x = np.concatenate([
    [start_xy[0]],
    wilde_out_df["latent_pca_1"].to_numpy(),
    wilde_return_df["latent_pca_1"].to_numpy(),
    [return_xy[0]],
])

connector_y = np.concatenate([
    [start_xy[1]],
    wilde_out_df["latent_pca_2"].to_numpy(),
    wilde_return_df["latent_pca_2"].to_numpy(),
    [return_xy[1]],
])

fig_latent.add_trace(go.Scattergl(
    x=connector_x,
    y=connector_y,
    mode="lines",
    line=dict(
        width=7,
        color="rgba(120, 0, 0, 0.55)",
    ),
    name="Full wilde connector",
    hoverinfo="skip",
))

# Show time-step clouds for the two music anchors
# These are the actual direct PCA positions of each latent time step.

fig_latent.add_trace(go.Scattergl(
    x=return_pca_steps[:, 0],
    y=return_pca_steps[:, 1],
    mode="markers",
    marker=dict(
        size=2.5,
        color="black",
        opacity=0.75,
    ),
    name="Wild start/return anchor; one time step",
    hovertemplate=(
        "Wild return anchor time step<br>"
        "latent_pca_1=%{x:.3f}<br>"
        "latent_pca_2=%{y:.3f}<extra></extra>"
    ),
))


# Wilde path rows
wilde_out_df = path_df[
    path_df["point_type"] == "wilde_out"
].sort_values("path_index").copy()

wilde_return_df = path_df[
    path_df["point_type"] == "wilde_return"
].sort_values("path_index").copy()

if len(wilde_out_df) == 0:
    raise RuntimeError("No wilde_out rows found in path_df.")

if len(wilde_return_df) == 0:
    raise RuntimeError("No wilde_return rows found in path_df.")


# For visible dots: show only 0.1 ... 0.9, not 1.0
# 1.0 is the wild target / final return anchor region.
wilde_out_dots_df = wilde_out_df[
    wilde_out_df["wilde_t"] < 1.0
].copy()

wilde_return_dots_df = wilde_return_df[
    wilde_return_df["wilde_t"] < 1.0
].copy()


# Brown connector underneath
# start anchor mean → wilde_out → wilde_return → return anchor mean
connector_x = np.concatenate([
    [start_xy[0]],
    wilde_out_df["latent_pca_1"].to_numpy(),
    wilde_return_df["latent_pca_1"].to_numpy(),
    [return_xy[0]],
])

connector_y = np.concatenate([
    [start_xy[1]],
    wilde_out_df["latent_pca_2"].to_numpy(),
    wilde_return_df["latent_pca_2"].to_numpy(),
    [return_xy[1]],
])

fig_latent.add_trace(go.Scattergl(
    x=connector_x,
    y=connector_y,
    mode="lines",
    line=dict(
        width=7,
        color="rgba(120, 70, 20, 0.55)",  # brown
    ),
    name="Full wilde connector",
    hoverinfo="skip",
))


# Red outward dots: wilde_t = 0.1 ... 0.9

fig_latent.add_trace(go.Scattergl(
    x=wilde_out_dots_df["latent_pca_1"],
    y=wilde_out_dots_df["latent_pca_2"],
    mode="markers",
    marker=dict(
        size=4,
        color="red",
        symbol="circle",
    ),
    name="Wilde outward points",
    customdata=np.stack([
        wilde_out_dots_df["path_index"],
        wilde_out_dots_df["wilde_t"],
        wilde_out_dots_df["latent_pca_radius"],
        wilde_out_dots_df["window_index"],
    ], axis=-1),
    hovertemplate=(
        "Wilde outward<br>"
        "path_index=%{customdata[0]}<br>"
        "wilde_t=%{customdata[1]:.2f}<br>"
        "latent_pca_radius=%{customdata[2]:.3f}<br>"
        "window_index=%{customdata[3]}<br>"
        "latent_pca_1=%{x:.3f}<br>"
        "latent_pca_2=%{y:.3f}<extra></extra>"
    ),
))

# Orange return dots: return_t = 0.1 ... 0.9

fig_latent.add_trace(go.Scattergl(
    x=wilde_return_dots_df["latent_pca_1"],
    y=wilde_return_dots_df["latent_pca_2"],
    mode="markers",
    marker=dict(
        size=4,
        color="orange",
        symbol="diamond",
    ),
    name="Return from wilde points",
    customdata=np.stack([
        wilde_return_dots_df["path_index"],
        wilde_return_dots_df["wilde_t"],
        wilde_return_dots_df["latent_pca_radius"],
        wilde_return_dots_df["window_index"],
    ], axis=-1),
    hovertemplate=(
        "Return from wilde<br>"
        "path_index=%{customdata[0]}<br>"
        "return_t=%{customdata[1]:.2f}<br>"
        "latent_pca_radius=%{customdata[2]:.3f}<br>"
        "window_index=%{customdata[3]}<br>"
        "latent_pca_1=%{x:.3f}<br>"
        "latent_pca_2=%{y:.3f}<extra></extra>"
    ),
))

# Anchor mean markers


fig_latent.add_trace(go.Scattergl(
    x=[return_xy[0]],
    y=[return_xy[1]],
    mode="markers",
    marker=dict(
        size=15,
        color="purple",
        symbol="star",
    ),
    name="Wild start/return anchor (mean)",
    hovertemplate=(
        "Wild return anchor mean<br>"
        f"metadata index={wild_return_idx}<br>"
        "latent_pca_1=%{x:.3f}<br>"
        "latent_pca_2=%{y:.3f}<extra></extra>"
    ),
))

fig_latent.update_layout(
    legend_title_text="Source / path",
)

fig_latent.show()

In [ ]:
# Join decoded segments with equal-power crossfade
song = song_segments[0].copy()

for segment in song_segments[1:]:

    fade = min(crossfade_samples, len(song), len(segment))

    t = np.linspace(0, 1, fade)

    fade_out = np.cos(t * np.pi / 2)
    fade_in = np.sin(t * np.pi / 2)

    overlap = song[-fade:] * fade_out + segment[:fade] * fade_in

    song = np.concatenate([
        song[:-fade],
        overlap,
        segment[fade:],
    ])

print("\nFinal song assembled.")

In [ ]:
out_wav = OUTPUT_DIR / "generated_song.wav"

sf.write(str(out_wav), song, SR)

print("\nSaved:", out_wav)

display(Markdown("## Generated song"))
display(Audio(song, rate=SR))

00-53s: 4 Nightjar anchors <br>
53s-1.10s: transition nightjar to similar songs<br>
1.10-2.03: similar songs<br>
2.03-2.20: transition similar songs to nightjar<br>
2.20-2.38: nightjar anchors<br>
2.38-3.13: walking away from last nightjar anchor<br>
3.13-3.48: walking back to nightjar anchor<br>
3.48-4.06: transition: nightjar to birds<br>
4.06-4.33: birds <br>

## TODOS:

## 1. SONG

- improve the generated song by choosing suitable manual anchors (some are more noisy than others); experiment with similar songs too (or random choice??)
- add birds to as anchors for beginning and ending

- "move into the wild": we have to add something more experimental (even if it sounds bad). She proposed that we move from the anchors to the outer latent space
    - idea: after last music anchor, we move to the center of the latent space, then we continue that direction until we are somewhere at the outer part of the point cloud ?

- generate plots:
    - live plot to music : what are the anchors? what are the new generated points? visualize graph lines
    - use summarized latent space for better visualization


- generate the final nightjar song. for the project it has to be of the form:
    - bird sounds intro ("human")
    - original nightjar song ("human")
    - ai generated nightjar song ("ai-generated")
        - start with bird sounds
        - go to nightjar music , maybe some similar songs
        - back to birds/nature

## 2. RAVE
For notebook & presentation:
- research about RAVE and briefly explain what it stands for, how it works, why it suited for our project
- RAVE = variational autoencoder-style audio autoencoder + adversarial fine-tuning ?
- if possible, add a plot for the rave model to show how the architecture looks like


## 3. LATENT PLOTS
- add more visualizations, eg for other priciple components (or 3d plot)

## 3. PRESENTATION

- create the powerpoint


## 4. NOTEBOOK
- add/improve my comments/descriptions/interpretations